In [1]:
import polars as pl

In [2]:
df = pl.scan_parquet('../data/train_full.parquet').sort(['customer_id', 'event_dttm'])

## Временные лаги
**Цель:** Выявление резких изменений в поведении клиента. 
Мошенники часто действуют быстро и меняют паттерны использования.

**Генерируемые признаки:**
* `time_since_last_op_sec` — время в секундах с момента предыдущей транзакции (поиск быстрых серий).
* `amt_ratio_to_prev` — отношение текущей суммы к предыдущей (поиск аномальных скачков трат).
* `is_mcc_changed` — флаг смены категории магазина (MCC).
* `is_device_changed` — флаг смены операционной системы (маркер входа с чужого устройства).
* `is_timezone_changed` — флаг смены часового пояса (маркер удаленной работы мошенника).

In [3]:
df = df.with_columns(
    pl.col('operaton_amt').fill_null(0.0)
)

df = df.with_columns(
    prev_dttm = pl.col('event_dttm').shift(1).over('customer_id'),
    prev_amt = pl.col('operaton_amt').shift(1).over('customer_id'),
    prev_mcc = pl.col('mcc_code').shift(1).over('customer_id'),
    prev_timezone = pl.col('timezone').shift(1).over('customer_id'),
    prev_os = pl.col('operating_system_type').shift(1).over('customer_id')
)

df = df.with_columns(
    time_since_last_op_sec = (
        pl.col('event_dttm') - pl.col('prev_dttm')
    ).dt.total_seconds(),
    amt_ratio_to_prev = pl.col('operaton_amt') / (pl.col('prev_amt') + 0.1),
    is_mcc_changed = pl.when(pl.col('mcc_code') != pl.col('prev_mcc')).then(pl.lit(1)).otherwise(pl.lit(0)),
    is_timezone_changed = pl.when(pl.col('timezone') != pl.col('prev_timezone')).then(pl.lit(1)).otherwise(pl.lit(0)),
    is_device_changed = pl.when(pl.col('operating_system_type') != pl.col('prev_os')).then(pl.lit(1)).otherwise(pl.lit(0))
).drop(['prev_dttm', 'prev_amt', 'prev_mcc', 'prev_timezone', 'prev_os'])

## Профиль клиента
**Цель:** Создание исторических профилей поведения клиентов без заглядывания в будущее.

Генерируемые признаки:
1. `client_op_seq_num` — порядковый номер транзакции клиента (счетчик активности).
2. `client_expanding_mean_amt` — накопительная средняя сумма транзакций клиента строго до момента текущей операции.
3. `amt_diff_from_exp_mean` — абсолютное отклонение текущей суммы операции от исторической нормы клиента.

In [4]:
df = df.with_columns(
    client_op_seq_num = pl.col('event_dttm').cum_count().over('customer_id')
).with_columns(
    client_expanding_mean_amt = pl.col('operaton_amt').cum_sum().over('customer_id') / pl.col('client_op_seq_num')
).with_columns(
    amt_diff_from_exp_mean = pl.col('operaton_amt') - pl.col('client_expanding_mean_amt')
)

In [5]:
df.head().collect()

customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,timezone,session_id,operating_system_type,battery,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,screen_w,screen_h,event_date,target,time_since_last_op_sec,amt_ratio_to_prev,is_mcc_changed,is_timezone_changed,is_device_changed,client_op_seq_num,client_expanding_mean_amt,amt_diff_from_exp_mean
i64,i64,datetime[μs],i8,i16,i8,i16,f32,i16,i8,i16,i16,i64,i8,i8,i8,i8,i8,str,i16,i16,date,i8,i64,f32,i32,i32,i32,u32,f64,f64
123123123123129,123999300382879,2024-10-01 05:29:14,14,75,6,5,56422.0,0,4,3,null,null,null,null,null,null,null,null,null,null,2024-10-01,0,null,null,0,0,0,1,56422.0,0.0
123123123123129,124531875713936,2024-10-01 10:17:22,7,56,4,15,0.0,null,null,null,null,null,null,null,null,null,null,null,null,null,2024-10-01,0,17288,0.0,0,0,0,2,28211.0,-28211.0
123123123123129,123329285580171,2024-10-01 10:20:03,3,120,6,5,300870.0,0,10,3,null,null,null,null,null,null,null,null,null,null,2024-10-01,0,161,3.0087e6,0,0,0,3,119097.333333,181772.666667
123123123123129,124334305430665,2024-10-02 07:48:09,14,75,6,5,298458.0,0,1,3,null,null,null,null,null,null,null,null,null,null,2024-10-02,0,77286,0.991983,1,0,0,4,163937.5,134520.5
123123123123129,126215501146513,2024-10-02 11:20:40,14,75,6,5,59944.0,0,15,3,null,null,null,null,null,null,null,null,null,null,2024-10-02,0,12751,0.200846,1,0,0,5,143138.8,-83194.8


In [6]:
df.sink_parquet('../data/train_features_v1.parquet')